In [2]:
# 핵심: 각 노트북은 독립 실행되므로 원본 CSV를 다시 읽어 동일한 전처리 출발점을 만든다.
from pathlib import Path

import pandas as pd

data_file = Path("data/house_tiny.csv")

# 01번 실습에서 만든 CSV 파일을 불러온다.
if not data_file.exists():
    raise FileNotFoundError("먼저 01_reading_the_dataset.ipynb를 실행하세요.")

data = pd.read_csv(data_file)

print("data.type:", type(data))
print(data)

data.type: <class 'pandas.DataFrame'>
   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


In [7]:
# 앞의 두 열은 모델 입력, 마지막 Price 열은 예측할 목표값이다.
# 핵심: 지도학습에서는 입력 특징 inputs와 예측 대상 targets를 분리하며 iloc의 stop 열은 포함되지 않는다.
inputs = data.iloc[:, 0:2]
targets = data.iloc[:, 2]

print("Inputs:")
print(inputs)
print("Shape:", inputs.shape)

print("\nTargets:")
print(targets)
print("Shape:", targets.shape)

assert inputs.shape == (4, 2)
assert targets.shape == (4,)

Inputs:
   NumRooms RoofType
0       NaN      NaN
1       2.0      NaN
2       4.0    Slate
3       NaN      NaN
Shape: (4, 2)

Targets:
0    127500
1    106000
2    178100
3    140000
Name: Price, dtype: int64
Shape: (4,)


In [9]:
# 범주형 RoofType을 모델이 처리할 수 있는 dummy 변수로 변환한다.
# dummy_na=True는 결측값도 하나의 독립적인 범주로 만든다.
# 핵심: one-hot encoding은 범주를 수치 열로 펼치고 dummy_na=True는 결측 자체도 하나의 범주로 보존한다.
inputs = pd.get_dummies(inputs, dummy_na=True)

print("After one-hot encoding:")
print(inputs)

print("\nColumns:")
print(inputs.columns.tolist())

assert "RoofType_Slate" in inputs.columns
assert "RoofType_nan" in inputs.columns
assert inputs.shape == (4, 3)

After one-hot encoding:
   NumRooms  RoofType_Slate  RoofType_nan
0       NaN           False          True
1       2.0           False          True
2       4.0            True         False
3       NaN           False          True

Columns:
['NumRooms', 'RoofType_Slate', 'RoofType_nan']


In [11]:
# 수치형 열만 선택해 결측값을 각 열의 평균으로 대체한다.
# 핵심: 평균 대치 통계는 실제 학습에서 훈련 세트로만 계산해야 검증·테스트 정보 누출을 막을 수 있다.
numeric_columns = inputs.select_dtypes(include="number").columns

inputs[numeric_columns] = inputs[numeric_columns].fillna(
    inputs[numeric_columns].mean()
)

print("After filling missing numerical values:")
print(inputs)

print("\nMissing values:")
print(inputs.isna().sum())

After filling missing numerical values:
   NumRooms  RoofType_Slate  RoofType_nan
0       3.0           False          True
1       2.0           False          True
2       4.0            True         False
3       3.0           False          True

Missing values:
NumRooms          0
RoofType_Slate    0
RoofType_nan      0
dtype: int64


In [12]:
# 핵심: 전처리 결과는 모든 값이 수치형이고 NaN이 없어야 하며 inputs와 targets의 첫 축 크기는 같아야 한다.
print("Prepared inputs:")
print(inputs)

print("\nPrepared targets:")
print(targets)

print("\nFinal shapes:")
print("inputs:", inputs.shape)
print("targets:", targets.shape)

Prepared inputs:
   NumRooms  RoofType_Slate  RoofType_nan
0       3.0           False          True
1       2.0           False          True
2       4.0            True         False
3       3.0           False          True

Prepared targets:
0    127500
1    106000
2    178100
3    140000
Name: Price, dtype: int64

Final shapes:
inputs: (4, 3)
targets: (4,)
